In [1]:
import re, json                        # re = text pattern shodhnyasaathi, json = chunks save karnyasaathi
from pathlib import Path               # folder/file paths sopya padhatine
from pypdf import PdfReader            # PDF madhun text kaadhto

DOCS_DIR = Path("../rag/docs")         # PDFs ithe aahet
INDEX_DIR = Path("../rag/index")       # chunks ithe save karnar
INDEX_DIR.mkdir(parents=True, exist_ok=True)   # folder nasel tar banvto

In [2]:
# he sagle patterns PDF madhle repeat hone-vale faltu text aahet
NOISE_PATTERNS = [
    r"¿Prefieres leer en español\?.*?página\.",                 # NKF Spanish banner
    r"Prefer to read in English\?.*?top menu\.",                # NKF English banner
    r"One day, no one.s life will be lost to kidney disease\..*?transplant access\.",  # NKF donation block
    r"\d{1,2}/\d{1,2}/\d{2}, \d{1,2}:\d{2} [AP]M .*? \| CDC",   # CDC print header (date + title)
    r"https://www\.cdc\.gov/\S+ \d+/\d+",                       # CDC print footer (URL + page no.)
    r"A Levin et al\.: Executive summary of KDIGO 2024 CKD Guideline",  # KDIGO page header
    r"KDIGO executive conclusions",                             # KDIGO page header
    r"Kidney International \(2024\) 105, 684.701",              # KDIGO page footer
]

def clean_text(text):
    text = re.sub(r"\s+", " ", text)                  # sagle newlines/extra spaces ek space madhe
    for p in NOISE_PATTERNS:
        text = re.sub(p, " ", text, flags=re.DOTALL)  # prattek faltu pattern kaadhto
    return re.sub(r"\s+", " ", text).strip()          # shevti parat spaces clean

In [3]:
def load_pages(pdf_path):
    reader = PdfReader(pdf_path)
    pages, stop = [], False
    for i, page in enumerate(reader.pages, start=1):   # page number 1 pasun
        raw = page.extract_text() or ""
        # KDIGO madhe "DISCLOSURE" nantar conflict-of-interest ani references aahet, te upayogi nahit
        if "DISCLOSURE" in raw:
            raw = raw.split("DISCLOSURE")[0]           # DISCLOSURE aadhicha text thevto
            stop = True                                # pudhche pages vachaycha nahit
        txt = clean_text(raw)
        if txt:                                        # rikama page skip
            pages.append({"page": i, "text": txt})
        if stop:
            break
    return pages

In [4]:
def split_text(text, max_chars=900, overlap=150):
    chunks, start = [], 0
    while start < len(text):
        end = min(start + max_chars, len(text))        # tukda max 900 characters
        if end < len(text):
            cut = text.rfind(". ", start, end)         # shevtcha purna vakya shodhto
            if cut > start + max_chars * 0.5:          # vakya khup aadhi sampat nasel tar tithe kapto
                end = cut + 1
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)          # 150 chars maage jaato, mhanje overlap
    return chunks

In [5]:
all_chunks = []
for pdf in sorted(DOCS_DIR.glob("*.pdf")):             # rag/docs madhle sagle PDFs
    n = 0
    pages = load_pages(pdf)
    for p in pages:
        for c in split_text(p["text"]):
            # prattek chunk sobat source ani page (citation saathi)
            all_chunks.append({"source": pdf.name, "page": p["page"], "text": c})
            n += 1
    print(f"{pdf.name[:55]:55s} pages={len(pages):2d} chunks={n}")

print("TOTAL chunks:", len(all_chunks))
lens = [len(c["text"]) for c in all_chunks]
print("min/avg/max length:", min(lens), sum(lens)//len(lens), max(lens))

6-Step Guide (NKF).pdf                                  pages= 5 chunks=10
CDC Basics.pdf                                          pages= 3 chunks=8
CDC Living with CKD.pdf                                 pages= 2 chunks=5
CDC Risk Factors.pdf                                    pages= 2 chunks=3
CDC Testing.pdf                                         pages= 2 chunks=8
CKD Stages G1-G5 (NKF).pdf                              pages= 7 chunks=8
eGFR (NKF).pdf                                          pages= 9 chunks=14
KDIGO 2024 Executive Summary.pdf                        pages=17 chunks=134
TOTAL chunks: 190
min/avg/max length: 174 741 900


In [6]:
# faltu text rahila ka te tapasto (sagle 0 pahije)
for kw in ["Prefieres", "Prefer to read", "Your gift today", "| CDC", "consultancy fees"]:
    print(kw, "->", sum(kw in c["text"] for c in all_chunks))

# ek sample chunk baghto
print(all_chunks[0])

# chunks file madhe save, pudhchya steps madhe parat PDFs vachayche nahit
with open(INDEX_DIR / "chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

Prefieres -> 0
Prefer to read -> 0
Your gift today -> 0
| CDC -> 0
consultancy fees -> 0
{'source': '6-Step Guide (NKF).pdf', 'page': 1, 'text': '6-Step Guide to Protecting Kidney Health Learn more about kidney disease, your risk, and how to prevent it. About 1 in 7 adults in the Unites States have chronic kidney disease (CKD). About 90% of adults with CKD do not know they have it! Kidney disease is a major public health concern. There are usually no noticeable symptoms in early kidney disease. Therefore, kidney disease often goes undetected until it is very advanced. Unfortunately, this is when someone would need dialysis or a kidney transplant. The key is to find kidney disease before the trouble starts. Regular testing for everyone is important, but it is especially important for people at risk. Follow these 6 steps to learn more about kidney disease, your risk, and how to prevent it.'}
